# 01 · Exploración de datos

Perfilamiento descriptivo de la base de control de saldos para la investigación sobre pensión mutualizada. El análisis es exploratorio: no estima mortalidad ni evalúa solvencia actuarial.

**Privacidad:** los gráficos y reportes se construyen con estadísticas y agregados. No exportan observaciones individuales. Los reportes generados están excluidos de Git; compruebe permisos y riesgo de divulgación antes de compartirlos.

## 1. Configuración y carga

El lector acepta `Base_Proyecto.xlsx` en la raíz o en `data/raw/`. Los encabezados se normalizan para manejar espacios, tildes y la codificación de la fuente.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

ROOT_CANDIDATES = (Path.cwd(), Path.cwd().parent)
PROJECT_ROOT = next((path.resolve() for path in ROOT_CANDIDATES if (path / "src").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró la carpeta del proyecto con el paquete src/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import find_workbook, load_workbook
from src.actuarial.eligibility import (
    DEFAULT_ISC_THRESHOLDS,
    calculate_eligibility_scenarios,
    calculate_isc,
    eligibility_figures,
    isc_data_quality,
)
from src.data.profile import (
    CATEGORICAL_COLUMNS,
    DEMOGRAPHIC_COLUMNS,
    FINANCIAL_COLUMNS,
    categorical_columns,
    datetime_columns,
    iqr_outlier_summary,
    numeric_columns,
    numeric_summary,
    overview_table,
    safe_category_counts,
)
from src.visualization.eda import (
    binned_relationship,
    categorical_figures,
    completeness_figures,
    coverage_band,
    coverage_figure,
    distribution_figures,
    financial_correlation_figure,
    financial_status_histograms,
    financial_violin_figures,
    risk_hierarchy_figures,
    scatter_matrix_figure,
)
from src.utils.reporting import calculate_kpis, write_reports

DATA_PATH = find_workbook(PROJECT_ROOT)
df = load_workbook(DATA_PATH)
print(f"Fuente local: {DATA_PATH.name} | Registros: {len(df):,} | Variables: {df.shape[1]}")

## 2. Estadísticas generales y diccionario observado

Los tipos corresponden a los valores leídos por pandas; no implican validación de semántica, unidades o definiciones de negocio.

In [ ]:
variable_overview = overview_table(df)
numeric_vars = numeric_columns(df)
categorical_vars = categorical_columns(df)
date_vars = datetime_columns(df)

display(pd.DataFrame({
    "indicador": ["Registros", "Variables", "Numéricas", "Categóricas", "Fechas"],
    "valor": [len(df), df.shape[1], len(numeric_vars), len(categorical_vars), len(date_vars)],
}))
display(variable_overview)

## 3. Calidad y valores extremos

Los extremos se identifican con la regla exploratoria de 1.5 IQR. La regla es una alerta para revisión; no elimina ni corrige registros.

In [ ]:
missing_by_variable = variable_overview[["variable", "nulos", "porcentaje_nulos"]].sort_values(
    "porcentaje_nulos", ascending=False
)
outliers = iqr_outlier_summary(df, [*DEMOGRAPHIC_COLUMNS, *FINANCIAL_COLUMNS])
display(missing_by_variable)
display(outliers.sort_values("porcentaje_extremos", ascending=False))

quality_figures = completeness_figures(df)
for figure in quality_figures.values():
    display(figure)

## 4. Análisis demográfico

Se describen edad y expectativa de vida disponible en la fuente. Esta última no reemplaza una estimación de supervivencia ni una tabla de mortalidad validada.

In [ ]:
demographic_summary = numeric_summary(df, DEMOGRAPHIC_COLUMNS)
display(demographic_summary)

figures = dict(quality_figures)
for column in DEMOGRAPHIC_COLUMNS:
    for kind, figure in distribution_figures(df, column).items():
        figures[f"{column}_{kind}"] = figure
        display(figure)

if "sexo" in df:
    sex_counts = safe_category_counts(df, "sexo")
    display(sex_counts)
    sex_charts = categorical_figures(df, "sexo", label="Sexo")
    figures.update({f"sexo_{name}": figure for name, figure in sex_charts.items()})
    for figure in sex_charts.values():
        display(figure)

## 5. Análisis financiero

Las cifras monetarias mantienen las unidades de la fuente y se presentan en millones de pesos para facilitar su lectura. El histograma principal agrupa en intervalos lineales; la versión logarítmica usa rangos que aumentan por razón geométrica sobre un eje categórico (no una escala logarítmica continua). El capital cero se muestra en su propia categoría y los nulos no se grafican. Verifique moneda y fecha de valoración antes de interpretar saldos.

In [ ]:
financial_summary = numeric_summary(df, FINANCIAL_COLUMNS)
display(financial_summary)

for column in FINANCIAL_COLUMNS:
    column_figures = distribution_figures(
        df,
        column,
        logarithmic=column in {"capital", "capital_mas_bono", "capital_mes_anterior", "capital_necesario_sm"},
    )
    figures.update({f"{column}_{kind}": figure for kind, figure in column_figures.items()})
    for figure in column_figures.values():
        display(figure)

violin_figures = financial_violin_figures(df, FINANCIAL_COLUMNS)
figures.update(violin_figures)
for figure in violin_figures.values():
    display(figure)

status_histograms = financial_status_histograms(df)
figures.update(status_histograms)
for figure in status_histograms.values():
    display(figure)

correlation_figure = financial_correlation_figure(df, FINANCIAL_COLUMNS)
if correlation_figure is not None:
    figures["correlaciones_financieras"] = correlation_figure
    display(correlation_figure)

## 6. Riesgos y participación

Los conteos de categorías pequeñas se agrupan en los resúmenes para reducir el riesgo de divulgación. Los porcentajes describen la base y no representan probabilidades de transición.

In [ ]:
for column in ("riesgo", "riesgo_3", "descapitalizado", "desfinanciado"):
    if column not in df:
        continue
    display(safe_category_counts(df, column))
    category_charts = categorical_figures(df, column)
    figures.update({f"{column}_{name}": figure for name, figure in category_charts.items()})
    for figure in category_charts.values():
        display(figure)

risk_figures = risk_hierarchy_figures(df)
figures.update(risk_figures)
for figure in risk_figures.values():
    display(figure)

## 7. Relaciones entre variables

Los gráficos bivariados son mapas de calor de conteos en intervalos. No incluyen puntos individuales ni permiten reconstruir registros de la fuente.

In [ ]:
relationships = [
    ("edad", "expectativa_vida", "Edad vs expectativa de vida", False, False),
    ("edad", "capital", "Edad vs capital", False, True),
    ("capital", "descapitalizacion", "Capital vs descapitalización", True, False),
    ("capital_necesario_sm", "capital", "Capital necesario para 1 SM vs capital", True, True),
]
for x_column, y_column, title, log_x, log_y in relationships:
    figure = binned_relationship(
        df, x_column, y_column, title, logarithmic_x=log_x, logarithmic_y=log_y
    )
    if figure is not None:
        figures[f"relacion_{x_column}_{y_column}"] = figure
        display(figure)

matrix = scatter_matrix_figure(
    df, ["edad", "expectativa_vida", "capital", "capital_necesario_sm"]
)
if matrix is not None:
    figures["matriz_relaciones"] = matrix
    display(matrix)

## 8. Indicadores iniciales y cobertura experimental

La cobertura EDA se define como `capital / capital_necesario_sm`; es distinta del ISC de descapitalización, que usa `capital_mas_bono / capital_necesario_sm` y excluye Riesgo GPM y Sustitución-GPM conforme a la clasificación observada. No es una tasa de reemplazo, una prima ni un indicador de solvencia de la bolsa mutualizada.

Los umbrales de selección ISC son determinísticos, aplicados estrictamente como `ISC < umbral`; no representan probabilidades ni una validación definitiva de elegibilidad. Se comparan con la clasificación fuente `descapitalizacion > 0`. El capital necesario es el valor actuarial calculado externamente y observado en la base; este notebook no recalcula beneficiarios ni expectativa de vida. Los conteos son registros de la hoja, no necesariamente afiliados únicos.

In [ ]:
kpis = calculate_kpis(df)
display(pd.DataFrame([kpis]).T.rename(columns={0: "valor"}))

coverage = coverage_band(df)
coverage_table = coverage.value_counts().rename_axis("banda").reset_index(name="registros")
coverage_table["participacion_pct"] = coverage_table["registros"] / len(df) * 100
display(coverage_table)
figures["cobertura"] = coverage_figure(df)
display(figures["cobertura"])

df_isc = calculate_isc(df)
isc_diagnostics = isc_data_quality(df_isc)
isc_scenarios = calculate_eligibility_scenarios(
    df_isc, thresholds=DEFAULT_ISC_THRESHOLDS
)
display(pd.DataFrame([isc_diagnostics]).T.rename(columns={0: "valor"}))
display(isc_scenarios)

isc_charts = eligibility_figures(isc_scenarios)
figures.update(isc_charts)
for figure in isc_charts.values():
    display(figure)

## 9. Exportación del dashboard y el informe Markdown

El dashboard HTML incorpora Plotly localmente y puede abrirse sin conexión a internet. Las salidas se guardan en `reports/` y están excluidas de Git.

In [ ]:
reports_dir = PROJECT_ROOT / "reports"
html_report, markdown_report = write_reports(
    frame=df,
    figures=figures,
    source_name=DATA_PATH.name,
    reports_dir=reports_dir,
    isc_scenarios=isc_scenarios,
    isc_diagnostics=isc_diagnostics,
)
print(f"Dashboard interactivo: {html_report}")
print(f"Reporte Markdown: {markdown_report}")
print(f"Gráficos incluidos: {len(figures)}")

## Interpretación y límites

- Confirme con los responsables del dato el universo, la unidad de observación, las definiciones de descapitalización y las unidades monetarias.
- Los extremos estadísticos no se corrigen automáticamente.
- La expectativa de vida de la fuente no sustituye un análisis de supervivencia.
- El EDA no estima sostenibilidad financiera, tasas de mortalidad, rentabilidad futura ni derechos de beneficiarios.
- Revise las reglas de confidencialidad institucional antes de mover el HTML o Markdown fuera del entorno autorizado.